In [1]:
import torch 
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms

In [2]:
# Hyperparameters
input_size = 28
sequence_length = 28
num_classes = 10
batch_size = 64

# Dataset
train_dataset = torchvision.datasets.MNIST(
    root='./data',
    train=True,
    transform=transforms.ToTensor(),
    download=True
)

test_dataset = torchvision.datasets.MNIST(
    root='./data',
    train=False,
    transform=transforms.ToTensor()
)

# DataLoader
train_loader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

In [3]:
test_dataset


Dataset MNIST
    Number of datapoints: 10000
    Root location: ./data
    Split: Test
    StandardTransform
Transform: ToTensor()

## BUILD THE RNN

In [4]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super(RNN, self).__init__()
        
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        # x: (batch, 1, 28, 28)
        
        x = x.squeeze(1)  # (batch, 28, 28)
        
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)
        
        out, _ = self.rnn(x, h0)
        
        out = out[:, -1, :]   # last timestep
        
        out = self.fc(out)
        return out

In [5]:
input_size = 28
hidden_size = 128
num_layers = 2
num_classes = 10
learning_rate = 0.001

model = RNN(input_size, hidden_size, num_layers, num_classes)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

## TRAIN THE RNN

In [6]:
num_epochs = 3

for epoch in range(num_epochs):
    for images, labels in train_loader:
        
        outputs = model(images)   # 👈 RNN use ho raha hai yaha
        loss = criterion(outputs, labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

In [7]:
rnn_correct = 0
rnn_total = 0

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        
        rnn_total += labels.size(0)
        rnn_correct += (predicted == labels).sum().item()

rnn_acc = 100 * rnn_correct / rnn_total
print(f"RNN Accuracy: {rnn_acc:.2f}%")

RNN Accuracy: 95.95%


## CNN

## Build 

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class CNN_Model(nn.Module):
    def __init__(self):
        super(CNN_Model, self).__init__()
        
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        
        self.pool = nn.MaxPool2d(2, 2)
        
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)
    
    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))   # (28 → 14)
        x = self.pool(F.relu(self.conv2(x)))   # (14 → 7)
        
        x = x.view(-1, 64 * 7 * 7)
        
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        
        return x

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CNN_Model().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

## Training

In [10]:
epochs = 10

for epoch in range(epochs):
    model.train()
    total_loss = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {total_loss:.4f}")

Epoch [1/10], Loss: 163.1697
Epoch [2/10], Loss: 45.4578
Epoch [3/10], Loss: 31.2499
Epoch [4/10], Loss: 22.9559
Epoch [5/10], Loss: 18.6398
Epoch [6/10], Loss: 14.3168
Epoch [7/10], Loss: 10.1578
Epoch [8/10], Loss: 7.8145
Epoch [9/10], Loss: 7.7637
Epoch [10/10], Loss: 6.5194


## Evaluation

In [11]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"Test Accuracy: {accuracy:.2f}%")

Test Accuracy: 99.04%
